In [1]:
!pip install gradio -q

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Libraries

In [2]:
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import gradio as gr
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


## Class Names

In [3]:
CLASS_NAMES = [
    'ahmed_sharif_chaudhry',
    'aitzaz_ahsan',
    'asif_ali_zardari',
    'assad_umar',
    'bilawal_bhutto_zardari',
    'chaudhary_nisar',
    'chaudhary_pervaiz_elahi',
    'imran_khan',
    'ishaq_dar',
    'khurshid_shah',
    'maryam_nawaz',
    'maulana_fazl-ur-rehman',
    'nawaz_sharif',
    'rana_sanaullah',
    'shah_mehmood_quereshi',
    'shahbaz_sharif'
]

# Display names (clean)
DISPLAY_NAMES = [name.replace('_', ' ').title() for name in CLASS_NAMES]
NUM_CLASSES = len(CLASS_NAMES)
print(f'Total classes: {NUM_CLASSES}')
for n in DISPLAY_NAMES:
    print(f'  {n}')

Total classes: 16
  Ahmed Sharif Chaudhry
  Aitzaz Ahsan
  Asif Ali Zardari
  Assad Umar
  Bilawal Bhutto Zardari
  Chaudhary Nisar
  Chaudhary Pervaiz Elahi
  Imran Khan
  Ishaq Dar
  Khurshid Shah
  Maryam Nawaz
  Maulana Fazl-Ur-Rehman
  Nawaz Sharif
  Rana Sanaullah
  Shah Mehmood Quereshi
  Shahbaz Sharif


## Models

In [8]:
def load_resnet50(path, num_classes):
    model = models.resnet50(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model.to(device)

def load_efficientnet_b0(path, num_classes):
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model.to(device)

def load_vgg16(path, num_classes):
    model = models.vgg16(weights=None)
    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Sequential(
        nn.Dropout(0.5),
        nn.Linear(in_features, 512),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(512, num_classes)
    )
    model.load_state_dict(torch.load(path, map_location=device))
    model.eval()
    return model.to(device)


resnet_model  = load_resnet50('/content/drive/MyDrive/resnet50_best.pth', NUM_CLASSES)
effnet_model  = load_efficientnet_b0('/content/drive/MyDrive/EfficientNet-B0-Finetuned.pth', NUM_CLASSES)
vgg_model     = load_vgg16('/content/drive/MyDrive/VGG16-Final.pth', NUM_CLASSES)

MODELS = {
    'ResNet-50 (97%)':       resnet_model,
    'EfficientNet-B0 (87%)': effnet_model,
    'VGG-16':                vgg_model,
}


## Function

In [9]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

def predict(image, model_choice):
    if image is None:
        return "Pehle image upload karo!", {}

    # Preprocess
    img = Image.fromarray(image).convert('RGB')
    img_tensor = transform(img).unsqueeze(0).to(device)

    # Select model
    model = MODELS[model_choice]

    # Predict
    with torch.no_grad():
        outputs = model(img_tensor)
        probs   = torch.softmax(outputs, dim=1)[0]

    # Top 3 results
    top3_probs, top3_idx = torch.topk(probs, 3)

    # Confidence dict for gradio label
    confidences = {}
    for prob, idx in zip(top3_probs, top3_idx):
        name = DISPLAY_NAMES[idx.item()]
        confidences[name] = float(prob)

    # Main prediction
    top_name = DISPLAY_NAMES[top3_idx[0].item()]
    top_conf = float(top3_probs[0]) * 100

    result = f'Prediction: {top_name}\nConfidence: {top_conf:.2f}%'
    return result, confidences

print('Prediction function ready!')

Prediction function ready!


## Gradio Interface Launch

In [10]:
with gr.Blocks(title='Pakistani Politician Classifier', theme=gr.themes.Soft()) as demo:

    gr.Markdown("""
    # 🇵🇰 Pakistani Politician Image Classifier
    ### Project 2 — CNN Image Classification
    Upload a face image and select a model to identify the politician.
    """)

    with gr.Row():
        with gr.Column(scale=1):
            image_input = gr.Image(
                label='Upload Image',
                type='numpy'
            )
            model_dropdown = gr.Dropdown(
                choices=list(MODELS.keys()),
                value='ResNet-50 (97%)',
                label='Select Model'
            )
            predict_btn = gr.Button('Classify', variant='primary')

        with gr.Column(scale=1):
            text_output  = gr.Textbox(label='Result', lines=3)
            label_output = gr.Label(
                label='Top 3 Predictions with Confidence',
                num_top_classes=3
            )

    gr.Markdown("""
    ### Supported Politicians:
    Ahmed Sharif Chaudhry | Aitzaz Ahsan | Asif Ali Zardari | Assad Umar |
    Bilawal Bhutto Zardari | Chaudhary Nisar | Chaudhary Pervaiz Elahi |
    Imran Khan | Ishaq Dar | Khurshid Shah | Maryam Nawaz |
    Maulana Fazl-Ur-Rehman | Nawaz Sharif | Rana Sanaullah |
    Shah Mehmood Quereshi | Shahbaz Sharif
    """)

    predict_btn.click(
        fn=predict,
        inputs=[image_input, model_dropdown],
        outputs=[text_output, label_output]
    )

demo.launch(share=True, debug=False)


/tmp/ipykernel_3096/18004768.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title='Pakistani Politician Classifier', theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://008bfe19485c1f4c44.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
